# Vehicle Plate Recognition Continuation

This continuation notebook is fully self-contained. It does not import the local `plate_ocr` package. If someone downloads only this notebook, it can still:

- install missing Python packages
- download the OpenALPR benchmark if needed
- run the improved segmentation pipeline
- apply cluster-guided balanced sampling
- compare SVM, balanced SVM, KNN, decision tree, Naive Bayes, and perceptron
- save continuation results into `results_continuation/`


## 1. Setup And All Functions

The next cell bootstraps the environment and defines every function used in the continuation pipeline.

In [1]:
from __future__ import annotations

import csv
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

NOTEBOOK_NAME = "vehicle_plate_recognition_continuation.ipynb"

def find_root() -> Path:
    candidate = Path.cwd()
    while True:
        if (candidate / NOTEBOOK_NAME).exists():
            return candidate
        if candidate == candidate.parent:
            return Path.cwd()
        candidate = candidate.parent

ROOT = find_root()
for folder_name in ("data", "results_continuation", "paper"):
    (ROOT / folder_name).mkdir(parents=True, exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".matplotlib"))
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

REQUIRED_PACKAGES = {
    "joblib": "joblib",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "PIL": "Pillow",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
}

missing_packages = [pip_name for import_name, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(import_name) is None]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])

def ensure_openalpr_dataset(root: Path) -> Path:
    dataset_root = root / "data" / "openalpr-benchmarks" / "seg_and_ocr"
    if dataset_root.exists():
        return dataset_root

    data_dir = root / "data"
    zip_path = data_dir / "openalpr-benchmarks-main.zip"
    url = "https://github.com/openalpr/benchmarks/archive/refs/heads/master.zip"
    print("Downloading OpenALPR benchmark...")
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(data_dir)
    extracted_root = data_dir / "benchmarks-master"
    target_root = data_dir / "openalpr-benchmarks"
    if target_root.exists() and not (target_root / "seg_and_ocr").exists():
        shutil.rmtree(target_root)
    if not target_root.exists():
        extracted_root.rename(target_root)
    if zip_path.exists():
        zip_path.unlink()
    return target_root / "seg_and_ocr"

import joblib
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw, ImageFilter
from scipy import ndimage
from sklearn.base import clone
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import Perceptron
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, calinski_harabasz_score, confusion_matrix, davies_bouldin_score, silhouette_score
from sklearn.model_selection import GroupKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

try:
    from IPython.display import Image as NotebookImage, display
except Exception:
    NotebookImage = None
    def display(*args, **kwargs):
        return None

@dataclass(frozen=True)
class PlateSample:
    filename: str
    state: str
    label: str

    def image_path(self, image_dir: Path) -> Path:
        return image_dir / self.filename

@dataclass(frozen=True)
class CharacterBox:
    x0: int
    x1: int
    y0: int
    y1: int

    @property
    def width(self) -> int:
        return self.x1 - self.x0

    @property
    def height(self) -> int:
        return self.y1 - self.y0

@dataclass(frozen=True)
class PlateSegmentation:
    image_size: tuple[int, int]
    band_top: int
    band_bottom: int
    character_boxes: tuple[CharacterBox, ...]

@dataclass(frozen=True)
class PlatePrediction:
    filename: str
    state: str
    label: str
    predicted: str
    component_count: int

    @property
    def exact_match(self) -> bool:
        return self.label == self.predicted

    @property
    def length_match(self) -> bool:
        return len(self.label) == len(self.predicted)

def load_groundtruth(csv_path: Path, min_chars: int = 4, max_chars: int = 8) -> list[PlateSample]:
    rows: list[PlateSample] = []
    with csv_path.open(newline="", encoding="utf-8") as handle:
        reader = csv.reader(handle)
        for row in reader:
            if len(row) != 3:
                continue
            label = row[2].strip().upper()
            if min_chars <= len(label) <= max_chars:
                rows.append(PlateSample(filename=row[0], state=row[1], label=label))
    return rows

def load_rgb_image(image_path: Path) -> Image.Image:
    return Image.open(image_path).convert("RGB")

def otsu_threshold(values: np.ndarray) -> float:
    hist, _ = np.histogram(values.ravel(), bins=256, range=(0.0, 1.0))
    total = values.size
    sum_total = float(np.dot(np.arange(256), hist))
    sum_background = 0.0
    weight_background = 0.0
    best_variance = -1.0
    threshold = 0
    for level in range(256):
        weight_background += hist[level]
        if weight_background == 0:
            continue
        weight_foreground = total - weight_background
        if weight_foreground == 0:
            break
        sum_background += level * hist[level]
        mean_background = sum_background / weight_background
        mean_foreground = (sum_total - sum_background) / weight_foreground
        between = weight_background * weight_foreground * (mean_background - mean_foreground) ** 2
        if between > best_variance:
            best_variance = between
            threshold = level
    return threshold / 255.0

def build_ink_map(image: Image.Image, blur_radius: int = 9) -> np.ndarray:
    blurred = image.filter(ImageFilter.GaussianBlur(radius=blur_radius))
    rgb = np.asarray(image).astype(np.float32) / 255.0
    rgb_blur = np.asarray(blurred).astype(np.float32) / 255.0
    value = rgb.max(axis=2)
    local_difference = np.linalg.norm(rgb - rgb_blur, axis=2) / np.sqrt(3.0)
    return 0.8 * local_difference + 0.2 * (1.0 - value)

def binarize_ink_map(ink_map: np.ndarray) -> np.ndarray:
    threshold = otsu_threshold(ink_map)
    mask = (ink_map > threshold).astype(np.uint8) * 255
    mask_image = Image.fromarray(mask)
    mask_image = mask_image.filter(ImageFilter.MaxFilter(3)).filter(ImageFilter.MinFilter(3))
    return np.asarray(mask_image) > 0

def locate_text_band(mask: np.ndarray) -> tuple[int, int]:
    height = mask.shape[0]
    y0 = int(height * 0.20)
    y1 = int(height * 0.88)
    band = mask[y0:y1]
    row_score = band.mean(axis=1)
    threshold = max(row_score.max() * 0.40, row_score.mean())
    good_rows = np.where(row_score >= threshold)[0]
    if len(good_rows) == 0:
        return y0, y1
    return y0 + int(good_rows[0]), y0 + int(good_rows[-1]) + 1

def filter_character_components(mask: np.ndarray, band_top: int, band_bottom: int) -> tuple[CharacterBox, ...]:
    cropped = mask[band_top:band_bottom]
    labeled, _ = ndimage.label(cropped)
    objects = ndimage.find_objects(labeled)
    boxes: list[CharacterBox] = []
    for component_index, component in enumerate(objects, start=1):
        if component is None:
            continue
        rows, cols = component
        component_height = rows.stop - rows.start
        component_width = cols.stop - cols.start
        component_area = int((labeled[component] == component_index).sum())
        if component_height < cropped.shape[0] * 0.45:
            continue
        if component_width < 4 or component_width > cropped.shape[1] * 0.25:
            continue
        if component_area < 20:
            continue
        boxes.append(CharacterBox(x0=cols.start, x1=cols.stop, y0=band_top + rows.start, y1=band_top + rows.stop))
    boxes.sort(key=lambda item: item.x0)
    return tuple(boxes)

def split_wide_character_boxes(mask: np.ndarray, band_top: int, character_boxes: tuple[CharacterBox, ...], width_factor: float = 1.6, valley_window_factor: float = 0.35) -> tuple[CharacterBox, ...]:
    if len(character_boxes) <= 1:
        return character_boxes
    widths = [box.width for box in character_boxes]
    median_width = float(np.median(widths)) if widths else 0.0
    if median_width <= 0:
        return character_boxes
    split_boxes: list[CharacterBox] = []
    for box in character_boxes:
        if box.width <= median_width * width_factor:
            split_boxes.append(box)
            continue
        local_mask = mask[(box.y0 - band_top):(box.y1 - band_top), box.x0:box.x1]
        column_mass = local_mask.mean(axis=0)
        target_pieces = max(2, int(round(box.width / median_width)))
        radius = max(2, int(median_width * valley_window_factor))
        cut_points: list[int] = []
        for piece_index in range(1, target_pieces):
            target = int(box.width * piece_index / target_pieces)
            left = max(1, target - radius)
            right = min(box.width - 1, target + radius)
            cut_points.append(target if left >= right else left + int(np.argmin(column_mass[left:right])))
        previous = 0
        candidates: list[CharacterBox] = []
        for cut in cut_points + [box.width]:
            if cut - previous < 4:
                continue
            candidates.append(CharacterBox(x0=box.x0 + previous, x1=box.x0 + cut, y0=box.y0, y1=box.y1))
            previous = cut
        split_boxes.extend(candidates if len(candidates) > 1 else [box])
    split_boxes.sort(key=lambda item: item.x0)
    return tuple(split_boxes)

def segment_plate(image_path: Path, split_wide_components: bool = False) -> PlateSegmentation:
    image = load_rgb_image(image_path)
    ink_map = build_ink_map(image)
    mask = binarize_ink_map(ink_map)
    band_top, band_bottom = locate_text_band(mask)
    boxes = filter_character_components(mask, band_top, band_bottom)
    if split_wide_components:
        boxes = split_wide_character_boxes(mask[band_top:band_bottom], band_top, boxes)
    return PlateSegmentation(image_size=(image.width, image.height), band_top=band_top, band_bottom=band_bottom, character_boxes=boxes)

def extract_character_features(image_path: Path, segmentation: PlateSegmentation, size: tuple[int, int] = (20, 32)) -> np.ndarray:
    image = load_rgb_image(image_path).convert("L")
    gray = np.asarray(image).astype(np.float32) / 255.0
    features: list[np.ndarray] = []
    for box in segmentation.character_boxes:
        patch = gray[box.y0:box.y1, box.x0:box.x1]
        padded = np.pad(patch, ((2, 2), (2, 2)), constant_values=1.0)
        resized = Image.fromarray((padded * 255).astype("uint8")).resize(size)
        inverted = 1.0 - (np.asarray(resized).astype(np.float32) / 255.0)
        features.append(inverted.flatten())
    if not features:
        return np.empty((0, size[0] * size[1]), dtype=np.float32)
    return np.stack(features).astype(np.float32)

def draw_segmentation_preview(image_path: Path, segmentation: PlateSegmentation) -> Image.Image:
    image = load_rgb_image(image_path).copy()
    draw = ImageDraw.Draw(image)
    draw.rectangle([0, segmentation.band_top, image.width - 1, segmentation.band_bottom - 1], outline="red", width=2)
    for box in segmentation.character_boxes:
        draw.rectangle([box.x0, box.y0, box.x1, box.y1], outline="cyan", width=2)
    return image

def continuation_model_zoo() -> dict[str, object]:
    return {
        "svm": Pipeline([("scale", StandardScaler()), ("clf", SVC(kernel="rbf", C=4.0, gamma="scale"))]),
        "svm_balanced": Pipeline([("scale", StandardScaler()), ("clf", SVC(kernel="rbf", C=4.0, gamma="scale", class_weight="balanced"))]),
        "knn": Pipeline([("scale", StandardScaler()), ("clf", KNeighborsClassifier(n_neighbors=3))]),
        "decision_tree": DecisionTreeClassifier(max_depth=18, random_state=42),
        "naive_bayes": Pipeline([("scale", MinMaxScaler()), ("clf", GaussianNB())]),
        "perceptron": Pipeline([("scale", StandardScaler()), ("clf", Perceptron(max_iter=2500, tol=1e-3, class_weight="balanced", random_state=42))]),
    }

def split_samples(samples: list[PlateSample], test_size: float = 0.2, random_state: int = 42) -> tuple[list[PlateSample], list[PlateSample]]:
    train_samples, test_samples = train_test_split(samples, test_size=test_size, random_state=random_state)
    return list(train_samples), list(test_samples)

def build_aligned_character_dataset(samples: list[PlateSample], image_dir: Path, split_wide_components: bool = False) -> tuple[np.ndarray, np.ndarray, np.ndarray, list[PlateSegmentation], list[PlateSample]]:
    feature_rows: list[np.ndarray] = []
    labels: list[str] = []
    groups: list[str] = []
    segmentations: list[PlateSegmentation] = []
    aligned_samples: list[PlateSample] = []
    for sample in samples:
        image_path = sample.image_path(image_dir)
        segmentation = segment_plate(image_path, split_wide_components=split_wide_components)
        features = extract_character_features(image_path, segmentation)
        if len(segmentation.character_boxes) != len(sample.label):
            continue
        aligned_samples.append(sample)
        segmentations.append(segmentation)
        for feature, label in zip(features, sample.label):
            feature_rows.append(feature)
            labels.append(label)
            groups.append(sample.filename)
    return np.asarray(feature_rows, dtype=np.float32), np.asarray(labels), np.asarray(groups), segmentations, aligned_samples

def cross_validate_models(models: dict[str, object], X: np.ndarray, y: np.ndarray, groups: np.ndarray) -> list[dict[str, float | str]]:
    splitter = GroupKFold(n_splits=5)
    rows: list[dict[str, float | str]] = []
    for name, model in models.items():
        scores = cross_val_score(model, X, y, cv=splitter, groups=groups, scoring="accuracy", n_jobs=1)
        rows.append({"model": name, "cv_mean_accuracy": float(scores.mean()), "cv_std_accuracy": float(scores.std())})
    rows.sort(key=lambda item: item["cv_mean_accuracy"], reverse=True)
    return rows

def fit_models(models: dict[str, object], X: np.ndarray, y: np.ndarray) -> dict[str, object]:
    trained: dict[str, object] = {}
    for name, model in models.items():
        fitted = clone(model)
        fitted.fit(X, y)
        trained[name] = fitted
    return trained

def evaluate_aligned_subset(trained_models: dict[str, object], samples: list[PlateSample], image_dir: Path, split_wide_components: bool = False) -> list[dict[str, float | str]]:
    X_test, y_test, _, _, aligned_samples = build_aligned_character_dataset(samples, image_dir, split_wide_components=split_wide_components)
    if len(aligned_samples) == 0:
        return []
    rows: list[dict[str, float | str]] = []
    for name, model in trained_models.items():
        predictions = model.predict(X_test)
        rows.append({"model": name, "aligned_char_accuracy": float(accuracy_score(y_test, predictions))})
    return rows

def predict_plate(model: object, image_path: Path, split_wide_components: bool = False) -> tuple[str, PlateSegmentation]:
    segmentation = segment_plate(image_path, split_wide_components=split_wide_components)
    features = extract_character_features(image_path, segmentation)
    if len(features) == 0:
        return "", segmentation
    prediction = "".join(model.predict(features))
    return prediction, segmentation

def evaluate_plate_predictions(model: object, samples: list[PlateSample], image_dir: Path, split_wide_components: bool = False) -> tuple[list[PlatePrediction], dict[str, float]]:
    predictions: list[PlatePrediction] = []
    aligned_total = 0
    aligned_exact = 0
    aligned_char_total = 0
    aligned_char_correct = 0
    plate_exact_total = 0
    for sample in samples:
        predicted, _ = predict_plate(model, sample.image_path(image_dir), split_wide_components=split_wide_components)
        row = PlatePrediction(filename=sample.filename, state=sample.state, label=sample.label, predicted=predicted, component_count=len(predicted))
        predictions.append(row)
        if row.exact_match:
            plate_exact_total += 1
        if row.length_match:
            aligned_total += 1
            if row.exact_match:
                aligned_exact += 1
            aligned_char_total += len(sample.label)
            aligned_char_correct += sum(a == b for a, b in zip(sample.label, predicted))
    metrics = {
        "test_plate_count": float(len(predictions)),
        "segmentation_length_match_rate": float(aligned_total / max(1, len(predictions))),
        "plate_exact_accuracy_all": float(plate_exact_total / max(1, len(predictions))),
        "plate_exact_accuracy_length_matched": float(aligned_exact / max(1, aligned_total)),
        "char_accuracy_length_matched": float(aligned_char_correct / max(1, aligned_char_total)),
    }
    return predictions, metrics

def save_predictions_csv(predictions: list[PlatePrediction], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["filename", "state", "groundtruth", "predicted", "component_count", "length_match", "exact_match"])
        for item in predictions:
            writer.writerow([item.filename, item.state, item.label, item.predicted, item.component_count, item.length_match, item.exact_match])

def save_model_comparison(rows: list[dict[str, float | str]], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        return
    fieldnames = list(rows[0].keys())
    with output_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def save_metrics(metrics: dict[str, object], output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

def save_best_model(model: object, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, output_path)

def save_confusion_matrix_plot(model: object, X: np.ndarray, y: np.ndarray, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    predictions = model.predict(X)
    labels = sorted(set(y.tolist()))
    matrix = confusion_matrix(y, predictions, labels=labels)
    fig, ax = plt.subplots(figsize=(12, 12))
    display_matrix = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=labels)
    display_matrix.plot(ax=ax, xticks_rotation=90, colorbar=False)
    ax.set_title("Continuation Character-Level Confusion Matrix")
    fig.tight_layout()
    fig.savefig(output_path, dpi=200)
    plt.close(fig)

def save_segmentation_examples(samples: list[PlateSample], image_dir: Path, output_dir: Path, limit: int = 8, split_wide_components: bool = False) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    for index, sample in enumerate(samples):
        if index >= limit:
            break
        segmentation = segment_plate(sample.image_path(image_dir), split_wide_components=split_wide_components)
        preview = draw_segmentation_preview(sample.image_path(image_dir), segmentation)
        preview.save(output_dir / f"{Path(sample.filename).stem}_{sample.label}.png")

def augment_feature(feature: np.ndarray, mode: str) -> np.ndarray:
    image = Image.fromarray((255 * (1.0 - feature.reshape(32, 20))).astype("uint8"))
    if mode == "shift_left":
        canvas = Image.new("L", image.size, 255)
        canvas.paste(image.crop((1, 0, 20, 32)), (0, 0))
        image = canvas
    elif mode == "shift_right":
        canvas = Image.new("L", image.size, 255)
        canvas.paste(image.crop((0, 0, 19, 32)), (1, 0))
        image = canvas
    elif mode == "shift_up":
        canvas = Image.new("L", image.size, 255)
        canvas.paste(image.crop((0, 1, 20, 32)), (0, 0))
        image = canvas
    elif mode == "shift_down":
        canvas = Image.new("L", image.size, 255)
        canvas.paste(image.crop((0, 0, 20, 31)), (0, 1))
        image = canvas
    elif mode == "rot_m5":
        image = image.rotate(-5, fillcolor=255)
    elif mode == "rot_p5":
        image = image.rotate(5, fillcolor=255)
    else:
        raise ValueError(f"Unknown augmentation mode: {mode}")
    return 1.0 - (np.asarray(image).astype(np.float32) / 255.0).reshape(-1)

def cluster_guided_resample(X: np.ndarray, y: np.ndarray, groups: np.ndarray, target_per_class: int = 35) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    augmentation_modes = ["shift_left", "shift_right", "shift_up", "shift_down", "rot_m5", "rot_p5"]
    features = [feature for feature in X]
    labels = [label for label in y]
    group_ids = [group for group in groups]
    per_class: dict[str, list[tuple[np.ndarray, str]]] = defaultdict(list)
    for feature, label, group in zip(X, y, groups):
        per_class[str(label)].append((feature, str(group)))
    for label, items in per_class.items():
        required = max(0, target_per_class - len(items))
        if required == 0:
            continue
        item_features = np.asarray([item[0] for item in items], dtype=np.float32)
        if len(items) >= 6:
            cluster_count = min(3, len(items))
            clustering = KMeans(n_clusters=cluster_count, n_init=10, random_state=42)
            cluster_ids = clustering.fit_predict(item_features)
            buckets: dict[int, list[tuple[np.ndarray, str]]] = defaultdict(list)
            for item, cluster_id in zip(items, cluster_ids):
                buckets[int(cluster_id)].append(item)
            ordered_pool: list[tuple[np.ndarray, str]] = []
            for cluster_id in sorted(buckets, key=lambda key: len(buckets[key])):
                ordered_pool.extend(buckets[cluster_id])
        else:
            ordered_pool = items
        for augment_index in range(required):
            source_feature, source_group = ordered_pool[augment_index % len(ordered_pool)]
            augmented = augment_feature(source_feature, augmentation_modes[augment_index % len(augmentation_modes)])
            features.append(augmented)
            labels.append(label)
            group_ids.append(source_group)
    return np.asarray(features, dtype=np.float32), np.asarray(labels), np.asarray(group_ids)

def compute_cluster_diagnostics(X: np.ndarray, random_state: int = 42) -> dict[str, dict[str, float | int]]:
    if len(X) < 20:
        return {}
    reduced = PCA(n_components=min(32, X.shape[1]), random_state=random_state).fit_transform(X)
    diagnostics: dict[str, dict[str, float | int]] = {}
    clustering_specs = [
        ("kmeans_36", KMeans(n_clusters=36, n_init=10, random_state=random_state)),
        ("agglomerative_36", AgglomerativeClustering(n_clusters=36)),
        ("dbscan", DBSCAN(eps=9.0, min_samples=5)),
    ]
    for name, estimator in clustering_specs:
        predicted = estimator.fit_predict(reduced)
        unique_labels = {int(label) for label in predicted if label != -1}
        usable_mask = predicted != -1
        cluster_count = len(unique_labels)
        noise_count = int((predicted == -1).sum())
        if cluster_count < 2 or int(usable_mask.sum()) < 10:
            diagnostics[name] = {"cluster_count": cluster_count, "noise_count": noise_count}
            continue
        reduced_usable = reduced[usable_mask]
        predicted_usable = predicted[usable_mask]
        diagnostics[name] = {
            "cluster_count": cluster_count,
            "noise_count": noise_count,
            "silhouette_score": float(silhouette_score(reduced_usable, predicted_usable)),
            "davies_bouldin_index": float(davies_bouldin_score(reduced_usable, predicted_usable)),
            "calinski_harabasz_index": float(calinski_harabasz_score(reduced_usable, predicted_usable)),
        }
    return diagnostics

dataset_root = ensure_openalpr_dataset(ROOT)
image_dir = dataset_root / "usimages"
groundtruth_csv = dataset_root / "groundtruth.csv"
results_dir = ROOT / "results_continuation"

print("Workspace:", ROOT)
print("Dataset:", dataset_root)

Workspace: /content
Dataset: /content/data/openalpr-benchmarks/seg_and_ocr


## 2. Load Data

In [2]:
samples = load_groundtruth(groundtruth_csv, min_chars=4, max_chars=8)
train_samples, test_samples = split_samples(samples, test_size=0.2, random_state=42)

X_train, y_train, groups_train, _, aligned_train_samples = build_aligned_character_dataset(
    train_samples,
    image_dir,
    split_wide_components=True,
)
X_test_aligned, y_test_aligned, _, _, aligned_test_samples = build_aligned_character_dataset(
    test_samples,
    image_dir,
    split_wide_components=True,
)

dataset_summary = {
    "all_samples": len(samples),
    "train_samples": len(train_samples),
    "test_samples": len(test_samples),
    "aligned_train_samples": len(aligned_train_samples),
    "aligned_test_samples": len(aligned_test_samples),
    "train_character_count": int(len(X_train)),
    "aligned_test_character_count": int(len(X_test_aligned)),
}
print(json.dumps(dataset_summary, indent=2))

{
  "all_samples": 737,
  "train_samples": 589,
  "test_samples": 148,
  "aligned_train_samples": 199,
  "aligned_test_samples": 54,
  "train_character_count": 1239,
  "aligned_test_character_count": 326
}


## 3. Cluster-Guided Balanced Sampling

In [3]:
X_train_resampled, y_train_resampled, groups_train_resampled = cluster_guided_resample(
    X_train,
    y_train,
    groups_train,
    target_per_class=35,
)

print({
    "original_train_characters": int(len(X_train)),
    "resampled_train_characters": int(len(X_train_resampled)),
    "unique_classes": int(len(set(y_train_resampled.tolist()))),
})

{'original_train_characters': 1239, 'resampled_train_characters': 1619, 'unique_classes': 36}


## 4. Train Continuation Models

In [4]:
models = continuation_model_zoo()
comparison_rows = cross_validate_models(models, X_train_resampled, y_train_resampled, groups_train_resampled)
trained_models = fit_models(models, X_train_resampled, y_train_resampled)
holdout_rows = evaluate_aligned_subset(
    trained_models,
    test_samples,
    image_dir,
    split_wide_components=True,
)
holdout_by_name = {row["model"]: row["aligned_char_accuracy"] for row in holdout_rows}
for row in comparison_rows:
    row["aligned_holdout_char_accuracy"] = float(holdout_by_name.get(row["model"], 0.0))
comparison_rows

[{'model': 'svm',
  'cv_mean_accuracy': 0.7671322345590258,
  'cv_std_accuracy': 0.029097848830720325,
  'aligned_holdout_char_accuracy': 0.8742331288343558},
 {'model': 'svm_balanced',
  'cv_mean_accuracy': 0.7671168978707918,
  'cv_std_accuracy': 0.028640423262498122,
  'aligned_holdout_char_accuracy': 0.8773006134969326},
 {'model': 'knn',
  'cv_mean_accuracy': 0.6417350693612376,
  'cv_std_accuracy': 0.042273741943889255,
  'aligned_holdout_char_accuracy': 0.7791411042944786},
 {'model': 'perceptron',
  'cv_mean_accuracy': 0.6196102102581854,
  'cv_std_accuracy': 0.04904305867336139,
  'aligned_holdout_char_accuracy': 0.745398773006135},
 {'model': 'naive_bayes',
  'cv_mean_accuracy': 0.6195143914583167,
  'cv_std_accuracy': 0.03290371762083555,
  'aligned_holdout_char_accuracy': 0.745398773006135},
 {'model': 'decision_tree',
  'cv_mean_accuracy': 0.5621365213763967,
  'cv_std_accuracy': 0.033760655894327245,
  'aligned_holdout_char_accuracy': 0.6288343558282209}]

## 5. Evaluate Best Model And Save Results

In [5]:
best_model_name = comparison_rows[0]["model"]
best_model = trained_models[best_model_name]
plate_predictions, plate_metrics = evaluate_plate_predictions(
    best_model,
    test_samples,
    image_dir,
    split_wide_components=True,
)
cluster_diagnostics = compute_cluster_diagnostics(X_train_resampled)

metrics = {
    "dataset": {
        **dataset_summary,
        "resampled_train_character_count": int(len(X_train_resampled)),
        "target_per_class": 35,
    },
    "best_model": best_model_name,
    "cross_validation": comparison_rows,
    "plate_metrics": plate_metrics,
    "cluster_diagnostics": cluster_diagnostics,
}

save_model_comparison(comparison_rows, results_dir / "model_comparison.csv")
save_predictions_csv(plate_predictions, results_dir / "sample_predictions.csv")
save_metrics(metrics, results_dir / "metrics.json")
save_best_model(best_model, results_dir / "best_model.joblib")
save_confusion_matrix_plot(best_model, X_test_aligned, y_test_aligned, results_dir / "confusion_matrix.png")
save_segmentation_examples(test_samples, image_dir, results_dir / "segmentation_examples", split_wide_components=True)

print("Best continuation model:", best_model_name)
print(json.dumps(plate_metrics, indent=2))

Best continuation model: svm
{
  "test_plate_count": 148.0,
  "segmentation_length_match_rate": 0.36486486486486486,
  "plate_exact_accuracy_all": 0.22297297297297297,
  "plate_exact_accuracy_length_matched": 0.6111111111111112,
  "char_accuracy_length_matched": 0.8742331288343558
}


In [6]:
cluster_diagnostics

{'kmeans_36': {'cluster_count': 36,
  'noise_count': 0,
  'silhouette_score': 0.15883797407150269,
  'davies_bouldin_index': 1.8468931902276013,
  'calinski_harabasz_index': 66.27379608154297},
 'agglomerative_36': {'cluster_count': 36,
  'noise_count': 0,
  'silhouette_score': 0.14128465950489044,
  'davies_bouldin_index': 1.9375618569657627,
  'calinski_harabasz_index': 60.77740478515625},
 'dbscan': {'cluster_count': 1, 'noise_count': 0}}

## 6. Interpretation

This continuation notebook improves the project in three main ways:

- segmentation improves because wide merged components are split more intelligently
- recognition improves because cluster-guided augmentation reduces class imbalance and preserves more intra-class variation
- the perceptron baseline connects the continuation directly to the later lecture material